<a href="https://colab.research.google.com/github/rsekola/Projet_data/blob/master/Apprentissage_automatique_classification_par_apprentissage_supervis%C3%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import io

df = pd.read_csv('African_crises_dataset.csv')

buf = io.StringIO()
df.info(buf=buf)
print(buf.getvalue())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1059 entries, 0 to 1058
Data columns (total 14 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   country_number                   1059 non-null   int64  
 1   country_code                     1059 non-null   object 
 2   country                          1059 non-null   object 
 3   year                             1059 non-null   int64  
 4   systemic_crisis                  1059 non-null   int64  
 5   exch_usd                         1059 non-null   float64
 6   domestic_debt_in_default         1059 non-null   int64  
 7   sovereign_external_debt_default  1059 non-null   int64  
 8   gdp_weighted_default             1059 non-null   float64
 9   inflation_annual_cpi             1059 non-null   float64
 10  independence                     1059 non-null   int64  
 11  currency_crises                  1059 non-null   int64  
 12  inflation_crises    

In [10]:
!pip install ydata-profiling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.7/398.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: multimethod
    Found existing installation: multimethod 2.0.2
    Uninstalling multimethod-2.0.2:
      Successfully uninstalled multimethod-2.0.2
  Attempting uninstall: visions
    Found existing installation: visions 0.7.4
    Uninstalling visions-0.7.4:
      Successfully uninstalled visions-0.7.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas-profiling 3.2.0 requires visions[type_image_path]==0.7.4, but you have visions 0.8.1 which is incompatible.


In [11]:
from ydata_profiling import ProfileReport
import pandas as pd

df = pd.read_csv('African_crises_dataset.csv')

profile = ProfileReport(df, title="African Crises Dataset Profiling Report", explorative=True)

output_path = "profiling_report.html"
profile.to_file(output_path)

output_path

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 14/14 [00:01<00:00, 11.66it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

'profiling_report.html'

In [13]:
# Missing values
missing = df.isna().sum()

# Duplicates count
dup_count = df.duplicated().sum()

# Outliers detection using IQR for numeric columns
outliers = {}
for col in df.select_dtypes(include='number').columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers[col] = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()

info_buf = io.StringIO()
info_buf.write("done")

4

In [14]:
# Variable cible
target = "banking_crisis"

# Séparation variables explicatives / cible
X = df.drop(columns=[target])
y = df[target]

print("Variable cible :", target)
print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

Variable cible : banking_crisis
Dimensions de X : (1059, 13)
Dimensions de y : (1059,)


In [15]:
from sklearn.model_selection import train_test_split

# Division 80% train / 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Taille X_train :", X_train.shape)
print("Taille X_test  :", X_test.shape)
print("Taille y_train :", y_train.shape)
print("Taille y_test  :", y_test.shape)

Taille X_train : (847, 13)
Taille X_test  : (212, 13)
Taille y_train : (847,)
Taille y_test  : (212,)


In [17]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Séparer colonnes numériques et catégoriques
categorical_cols = X_train.select_dtypes(include=['object']).columns
numerical_cols = X_train.select_dtypes(include=['number']).columns

# Préprocessing
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numerical_cols)
    ]
)

# Pipeline complet
model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", RandomForestClassifier(n_estimators=200, random_state=42))
])

# Entraînement
model.fit(X_train, y_train)

# Prédictions
y_pred = model.predict(X_test)

# Évaluation
print("Accuracy :", accuracy_score(y_test, y_pred))
print("\nClassification Report :\n", classification_report(y_test, y_pred))
print("\nMatrice de confusion :\n", confusion_matrix(y_test, y_pred))


Accuracy : 0.9905660377358491

Classification Report :
               precision    recall  f1-score   support

      crisis       1.00      0.89      0.94        19
   no_crisis       0.99      1.00      0.99       193

    accuracy                           0.99       212
   macro avg       0.99      0.95      0.97       212
weighted avg       0.99      0.99      0.99       212


Matrice de confusion :
 [[ 17   2]
 [  0 193]]


In [18]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy :", accuracy)

# Classification report : précision, rappel, F1
print("\nClassification Report :\n", classification_report(y_test, y_pred))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
print("\nMatrice de confusion :\n", cm)

# Si c'est un problème binaire, calcule le ROC-AUC
if len(y_test.unique()) == 2:
    y_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    print("\nROC-AUC :", auc)


Accuracy : 0.9905660377358491

Classification Report :
               precision    recall  f1-score   support

      crisis       1.00      0.89      0.94        19
   no_crisis       0.99      1.00      0.99       193

    accuracy                           0.99       212
   macro avg       0.99      0.95      0.97       212
weighted avg       0.99      0.99      0.99       212


Matrice de confusion :
 [[ 17   2]
 [  0 193]]

ROC-AUC : 0.9953640578129261
